# MedGuard AI - Feature Engineering

In this notebook, we will engineer new features to help our machine learning models capture better patterns, and we will encode our categorical variables into numerical formats.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned dataset
df = pd.read_csv('../data/processed/cleaned_appointments.csv')
display(df.head())

,PatientID,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,No_Show,WaitingDays
0,2.987250e+13,5642903,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,0,0
1,5.589978e+14,5642503,M,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,0,0
2,4.262962e+12,5642549,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,0,0
3,8.679512e+11,5642828,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,0,0
4,8.841186e+12,5642494,F,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,0,0


### Step 1: Create Temporal Features from Dates
We can extract valuable information from the dates, such as the day of the week or the month. For example, people might be more likely to miss an appointment on a Friday than on a Tuesday.

In [2]:
# Ensure columns are datetime objects (since reading from CSV converts them back to strings)
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# Extract weekday names
df['Appointment_Weekday'] = df['AppointmentDay'].dt.day_name()
df['Scheduled_Weekday'] = df['ScheduledDay'].dt.day_name()

# Extract month names
df['Appointment_Month'] = df['AppointmentDay'].dt.month_name()

display(df[['AppointmentDay', 'Appointment_Weekday', 'Appointment_Month']].head())

,AppointmentDay,Appointment_Weekday,Appointment_Month
0,2016-04-29 00:00:00+00:00,Friday,April
1,2016-04-29 00:00:00+00:00,Friday,April
2,2016-04-29 00:00:00+00:00,Friday,April
3,2016-04-29 00:00:00+00:00,Friday,April
4,2016-04-29 00:00:00+00:00,Friday,April


### Step 2: Categorize Age into Groups
Instead of looking at every individual age (which can introduce noise), categorizing ages into life stages (Child, Adult, Senior) can sometimes reveal stronger, broader behavioral patterns.

In [3]:
# Define bins and labels for age categories
bins = [-1, 12, 64, 150] # -1 to 12 is Child, 13 to 64 is Adult, 65 to 150 is Senior
labels = ['Child', 'Adult', 'Senior']

# Use pd.cut to bucket the ages into our defined groups
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

display(df['AgeGroup'].value_counts())

AgeGroup
Adult     75085
Child     21035
Senior    14401
Name: count, dtype: int64

### Step 3: Encode Categorical Variables
Machine Learning algorithms only understand numbers. We must convert categorical text data (like Gender, Weekdays, and Neighbourhoods) into a format they can process, using **One-Hot Encoding**.

In [4]:
# We will use pd.get_dummies to one-hot encode these categorical columns.
# drop_first=True is used to avoid the dummy variable trap (perfect multicollinearity).
categorical_cols = ['Gender', 'Appointment_Weekday', 'Scheduled_Weekday', 'Appointment_Month', 'AgeGroup', 'Neighbourhood']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Convert boolean outputs from get_dummies to integers (1 and 0)
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'bool':
        df_encoded[col] = df_encoded[col].astype(int)

print(f"Original shape: {df.shape}")
print(f"Encoded shape: {df_encoded.shape}")

Original shape: (110522, 19)
Encoded shape: (110522, 108)


### Step 4: Analyze Feature Correlations
Now that all features are numeric, we can compute the Pearson correlation matrix to see which features correlate most strongly with our target variable, `No_Show`.

In [5]:
# Compute correlation matrix
corr = df_encoded.corr()

# Look at correlations specifically with our target 'No_Show'
target_corr = corr['No_Show'].sort_values(ascending=False)

print("Top 10 Positively Correlated Features (Increases likelihood of No-Show):")
print(target_corr.head(10))
print("\nTop 10 Negatively Correlated Features (Decreases likelihood of No-Show):")
print(target_corr.tail(10))

Top 10 Positively Correlated Features (Increases likelihood of No-Show):
No_Show                            1.000000
WaitingDays                        0.186322
SMS_received                       0.126505
AgeGroup_Adult                     0.029779
Scholarship                        0.029167
Neighbourhood_ITARARÉ              0.027433
Appointment_Month_May              0.024334
Neighbourhood_SANTOS DUMONT        0.023501
Neighbourhood_JESUS DE NAZARETH    0.017055
Neighbourhood_ILHA DO PRÍNCIPE     0.011850
Name: No_Show, dtype: float64

Top 10 Negatively Correlated Features (Decreases likelihood of No-Show):
Diabetes                        -0.015156
Neighbourhood_SANTA MARTHA      -0.018495
Neighbourhood_JARDIM DA PENHA   -0.018592
AppointmentDay                  -0.022327
Appointment_Month_June          -0.024214
Hypertension                    -0.035660
AgeGroup_Senior                 -0.045231
Age                             -0.060311
AppointmentID                   -0.162624
Sched

### Step 5: Save the Engineered Dataset
Before saving, we drop unique ID columns (`PatientID`, `AppointmentID`) and datetime objects, as these cannot be used directly in machine learning models and just introduce noise.

In [6]:
# Drop unneeded identifier and datetime columns
columns_to_drop = ['PatientID', 'AppointmentID', 'ScheduledDay', 'AppointmentDay']
df_encoded.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Save the final engineered dataset
output_path = '../data/processed/engineered_appointments.csv'
df_encoded.to_csv(output_path, index=False)
print(f"Successfully saved engineered dataset to {output_path}")

Successfully saved engineered dataset to ../data/processed/engineered_appointments.csv
